In [3]:
#import libraries and folders

import os, time
import pandas as pd
import numpy as np
import yfinance as yf

for folder in ["../data/raw", "../data/clean", "../data/powerbi"]:
    os.makedirs(folder, exist_ok=True)


In [4]:
#define assets to collect

assets = pd.DataFrame([
    # Asset, Ticker, Category, Transform
    ("S&P 500","^GSPC","Equity Index","log"), ("Nasdaq","^IXIC","Equity Index","log"),
    ("Dow Jones","^DJI","Equity Index","log"), ("FTSE 100","^FTSE","Equity Index","log"),
    ("DAX","^GDAXI","Equity Index","log"), ("Nikkei 225","^N225","Equity Index","log"),
    ("Hang Seng","^HSI","Equity Index","log"), ("Shanghai","000001.SS","Equity Index","log"),
    ("Thai SET","^SET.BK","Equity Index","log"), ("Emerging Mkts ETF","EEM","Equity Index","log"),
    ("Gold","GC=F","Commodity","log"), ("Silver","SI=F","Commodity","log"),
    ("Crude Oil WTI","CL=F","Commodity","log"), ("Brent Oil","BZ=F","Commodity","log"),
    ("Natural Gas","NG=F","Commodity","log"), ("Copper","HG=F","Commodity","log"),
    ("Wheat","ZW=F","Commodity","log"), ("Corn","ZC=F","Commodity","log"),
    ("Dollar Index","DX-Y.NYB","Currency","log"), ("EUR/USD","EURUSD=X","Currency","log"),
    ("USD/JPY","JPY=X","Currency","log"), ("GBP/USD","GBPUSD=X","Currency","log"),
    ("USD/THB","THB=X","Currency","log"), ("USD/CNY","CNY=X","Currency","log"),
    ("AUD/USD","AUDUSD=X","Currency","log"), ("USD/CHF","CHF=X","Currency","log"),
    ("Bitcoin","BTC-USD","Crypto","log"), ("Ethereum","ETH-USD","Crypto","log"),
    ("10Y Yield","^TNX","Rates","diff"), ("5Y Yield","^FVX","Rates","diff"),
    ("3M Yield","^IRX","Rates","diff"), ("Long Treasury ETF","TLT","Bonds","log"),
    ("High Yield Bonds","HYG","Bonds","log"), ("Corp Bonds","LQD","Bonds","log"),
    ("Tech","XLK","Sector","log"), ("Energy","XLE","Sector","log"),
    ("Financials","XLF","Sector","log"), ("Health","XLV","Sector","log"),
    ("Utilities","XLU","Sector","log"), ("Staples","XLP","Sector","log"),
    ("Discretionary","XLY","Sector","log"), ("Industrials","XLI","Sector","log"),
    ("Real Estate","VNQ","Sector","log"), ("VIX","^VIX","Volatility","diff"),
], columns=["Asset", "Ticker", "Category", "Transform"])

assets.to_csv("../data/clean/dim_asset.csv", index=False)
print(len(assets), "assets defined")
assets["Category"].value_counts()

44 assets defined


Category
Equity Index    10
Sector           9
Commodity        8
Currency         8
Rates            3
Bonds            3
Crypto           2
Volatility       1
Name: count, dtype: int64

In [5]:
#download data from Yahoo Finance

START = "2000-01-01"
BATCH_SIZE = 8

tickers = assets["Ticker"].tolist()
frames = []

for i in range(0, len(tickers), BATCH_SIZE):
    batch = tickers[i:i + BATCH_SIZE]
    print(f"Downloading batch {i // BATCH_SIZE + 1}: {batch}")
    try:
        data = yf.download(batch, start=START, auto_adjust=True, progress=False)
        close = data["Close"]
        if isinstance(close, pd.Series):         
            close = close.to_frame(batch[0])
        frames.append(close)
    except Exception as e:
        print("  Batch failed:", e)
    time.sleep(3)

raw = pd.concat(frames, axis=1)
raw.index = pd.to_datetime(raw.index).tz_localize(None)
raw = raw.sort_index()
print("\nShape:", raw.shape)


Shape: (8262, 44)


C:\Users\User\AppData\Local\Temp\ipykernel_1076\804926342.py:22: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  raw = pd.concat(frames, axis=1)


In [6]:
#check data completeness

check = pd.DataFrame({
    "First Date": raw.apply(lambda s: s.first_valid_index()),
    "Last Date": raw.apply(lambda s: s.last_valid_index()),
    "Rows": raw.notna().sum(),
})
check = check.reset_index().rename(columns={"index": "Ticker"})
check = check.merge(assets[["Ticker", "Asset", "Category"]], on="Ticker", how="right")

missing = check[check["Rows"].isna() | (check["Rows"] == 0)]
print("Tickers with NO data:")
print(missing[["Asset", "Ticker"]] if len(missing) else "None, all good")

check.sort_values("First Date")

Tickers with NO data:
None, all good


,Ticker,First Date,Last Date,Rows,Asset,Category
0,^GSPC,2000-01-03,2026-10-27,6745,S&P 500,Equity Index
41,XLI,2000-01-03,2026-10-27,6745,Industrials,Sector
40,XLY,2000-01-03,2026-10-27,6745,Discretionary,Sector
39,XLP,2000-01-03,2026-10-27,6745,Staples,Sector
38,XLU,2000-01-03,2026-10-27,6745,Utilities,Sector
37,XLV,2000-01-03,2026-10-27,6745,Health,Sector
36,XLF,2000-01-03,2026-10-27,6745,Financials,Sector
35,XLE,2000-01-03,2026-10-27,6745,Energy,Sector
34,XLK,2000-01-03,2026-10-27,6745,Tech,Sector
30,^IRX,2000-01-03,2026-10-27,6739,3M Yield,Rates


In [7]:
#save the raw data to parquaet file

raw = raw.dropna(axis=1, how="all")

raw.to_parquet("../data/raw/prices_raw.parquet")
print("Saved:", raw.shape)
print("File size (MB):", round(os.path.getsize("../data/raw/prices_raw.parquet") / 1e6, 2))

Saved: (8262, 44)
File size (MB): 1.77


In [8]:
#Set up FRED (macro data)
import os
from fredapi import Fred
from dotenv import load_dotenv

load_dotenv("../.env", override=True)
api_key = (os.environ.get("FRED_API_KEY") or "").strip().strip('"').strip("'")

if len(api_key) != 32:
    raise ValueError(f"FRED key should be 32 characters, found {len(api_key)}. Check .env")

fred = Fred(api_key=api_key)
print(fred.get_series("FEDFUNDS", observation_start="2024-01-01").tail(3))


2026-07-01    3.63
2026-08-01    3.63
2026-09-01    3.75
dtype: float64


In [9]:
#Download macroecoonomic data from FRED
macro_series = {
    "FEDFUNDS": "Fed Funds Rate",
    "CPIAUCSL": "CPI (Inflation Index)",
    "UNRATE": "Unemployment Rate",
    "M2SL": "M2 Money Supply",
    "T10Y2Y": "Yield Curve (10Y-2Y)",
    "DFII10": "Real 10Y Yield",
    "BAMLH0A0HYM2": "High Yield Spread",
    "USREC": "US Recession Flag",
}

macro = {}
for code, name in macro_series.items():
    try:
        macro[name] = fred.get_series(code, observation_start=START)
        print(f"OK   {name}: {macro[name].notna().sum()} rows")
    except Exception as e:
        print(f"FAIL {name}: {e}")

macro = pd.concat(macro, axis=1)
macro.index = pd.to_datetime(macro.index)
macro.index.name = "Date"
macro.tail()

OK   Fed Funds Rate: 321 rows
OK   CPI (Inflation Index): 320 rows
OK   Unemployment Rate: 320 rows
OK   M2 Money Supply: 321 rows
OK   Yield Curve (10Y-2Y): 6708 rows
OK   Real 10Y Yield: 5958 rows
OK   High Yield Spread: 784 rows
OK   US Recession Flag: 321 rows


C:\Users\User\AppData\Local\Temp\ipykernel_1076\68541655.py:21: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  macro = pd.concat(macro, axis=1)


,Fed Funds Rate,CPI (Inflation Index),Unemployment Rate,M2 Money Supply,Yield Curve (10Y-2Y),Real 10Y Yield,High Yield Spread,US Recession Flag
Date,,,,,,,,
2026-10-21,NaN,NaN,NaN,NaN,0.32,2.76,3.27,NaN
2026-10-22,NaN,NaN,NaN,NaN,0.31,2.64,3.40,NaN
2026-10-23,NaN,NaN,NaN,NaN,0.31,2.66,3.38,NaN
2026-10-26,NaN,NaN,NaN,NaN,0.32,2.66,3.30,NaN
2026-10-27,NaN,NaN,NaN,NaN,0.29,NaN,NaN,NaN


In [10]:
#Save the data

macro.to_parquet("../data/raw/macro_raw.parquet")

print("PRICES:", raw.shape, "|", raw.index.min().date(), "to", raw.index.max().date())
print("MACRO :", macro.shape, "|", macro.index.min().date(), "to", macro.index.max().date())
print("\nFiles in data/raw:", os.listdir("../data/raw"))

PRICES: (8262, 44) | 2000-01-03 to 2026-10-28
MACRO : (7101, 8) | 2000-01-01 to 2026-10-27

Files in data/raw: ['macro_raw.parquet', 'prices_raw.parquet']
